# Test de conversion des cultivars STICS V9 vers V11

Ce notebook convertit tous les fichiers maïs `maiplt*.txt` du jeu de test `tests/v9tov11`, puis vérifie que les valeurs suivent la priorité : **variété V9 → espèce V9 → valeur par défaut V11**.

## 1. Imports et chemins

In [1]:
from pathlib import Path
import sys

# Fonctionne depuis la racine du dépôt ou depuis le dossier notebooks.
repo_root = Path.cwd().resolve()
if not (repo_root / "src" / "modfilegen").is_dir():
    repo_root = repo_root.parent
if not (repo_root / "src" / "modfilegen").is_dir():
    raise FileNotFoundError("Impossible de trouver la racine du dépôt ModFileGen")

src_dir = repo_root / "src"
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

from modfilegen.utils.convert_stics_v9_cultivars_to_v11 import (
    as_values,
    convert,
    named_varieties,
    read_pairs,
    split_sections,
)

test_dir = repo_root / "tests" / "v9tov11" / "cultivars"
v9_dir = test_dir / "sticsv9"
template = test_dir / "template" / "ficplt1.txt"
output_dir = test_dir / "sticsv11"

sources = sorted(v9_dir.glob("maiplt*.txt"))
assert template.is_file(), template
assert sources, f"Aucun fichier maiplt*.txt dans {v9_dir}"
print(f"{len(sources)} fichiers maïs à convertir")
print(f"Modèle V11 : {template.relative_to(repo_root)}")
print(f"Sortie : {output_dir.relative_to(repo_root)}")

31 fichiers maïs à convertir
Modèle V11 : tests/v9tov11/cultivars/template/ficplt1.txt
Sortie : tests/v9tov11/cultivars/sticsv11


## 2. Conversion

In [2]:
results = {}
for source in sources:
    destination = output_dir / source.name
    results[source.name] = convert(source, template, destination)

for name, stats in results.items():
    print(
        f"{name}: {stats.varieties} variété(s), "
        f"{len(stats.move_specie_param_v9_to_v11)} paramètre(s) d'espèce V9 déplacé(s), "
        f"{len(stats.new_variety_param_v11)} nouveau(x) paramètre(s) V11 au niveau variété"
    )

maiplt.txt: 24 variété(s), 792 valeur(s) d'espèce V9 déplacée(s), 48 défaut(s) V11 au niveau variété
maiplt1_c1000.txt: 1 variété(s), 33 valeur(s) d'espèce V9 déplacée(s), 2 défaut(s) V11 au niveau variété
maiplt1_c1050.txt: 1 variété(s), 33 valeur(s) d'espèce V9 déplacée(s), 2 défaut(s) V11 au niveau variété
maiplt1_c1100.txt: 1 variété(s), 33 valeur(s) d'espèce V9 déplacée(s), 2 défaut(s) V11 au niveau variété
maiplt1_c1150.txt: 1 variété(s), 33 valeur(s) d'espèce V9 déplacée(s), 2 défaut(s) V11 au niveau variété
maiplt1_c1200.txt: 1 variété(s), 33 valeur(s) d'espèce V9 déplacée(s), 2 défaut(s) V11 au niveau variété
maiplt1_c1250.txt: 1 variété(s), 33 valeur(s) d'espèce V9 déplacée(s), 2 défaut(s) V11 au niveau variété
maiplt1_c1300.txt: 1 variété(s), 33 valeur(s) d'espèce V9 déplacée(s), 2 défaut(s) V11 au niveau variété
maiplt1_c1350.txt: 1 variété(s), 33 valeur(s) d'espèce V9 déplacée(s), 2 défaut(s) V11 au niveau variété
maiplt1_c1400.txt: 1 variété(s), 33 valeur(s) d'espèce V9 d

## 3. Validation exhaustive des valeurs et de la structure

In [ ]:
template_species_pairs, template_variety_blocks = split_sections(read_pairs(template), template)
template_varieties = named_varieties(template_variety_blocks)
first_template_block = next(iter(template_varieties.values()))

for source in sources:
    destination = output_dir / source.name
    source_species_pairs, source_variety_blocks = split_sections(read_pairs(source), source)
    output_species_pairs, output_variety_blocks = split_sections(read_pairs(destination), destination)

    source_species = as_values(source_species_pairs)
    output_species = as_values(output_species_pairs)
    source_varieties = named_varieties(source_variety_blocks)
    output_varieties = named_varieties(output_variety_blocks)

    assert list(output_species) == [name for name, _ in template_species_pairs]
    assert list(output_varieties) == list(source_varieties)

    for parameter, default_value in template_species_pairs:
        expected = source_species.get(parameter, default_value)
        assert output_species[parameter] == expected, (source.name, parameter)

    for variety_name, source_block in source_varieties.items():
        source_values = as_values(source_block[1:])
        output_values = as_values(output_varieties[variety_name][1:])
        template_block = template_varieties.get(variety_name, first_template_block)
        assert list(output_values) == [name for name, _ in template_block[1:]]

        for parameter, default_value in template_block[1:]:
            expected = source_values.get(parameter, source_species.get(parameter, default_value))
            assert output_values[parameter] == expected, (
                source.name, variety_name, parameter
            )

print(f"Validation réussie pour les {len(sources)} fichiers convertis.")

## 4. Noms des paramètres déplacés et nouveaux paramètres V11

In [ ]:
example_result = results["maiplt1_c850.txt"]

print("Paramètres déplacés d'espèce V9 vers variété V11 :")
print(*example_result.move_specie_param_v9_to_v11, sep="\n")
print("\nNouveaux paramètres d'espèce V11 :")
print(*example_result.new_specie_param_v11, sep="\n")
print("\nNouveaux paramètres de variété V11 :")
print(*example_result.new_variety_param_v11, sep="\n")